# Fraud Shield AI — Capstone Project

Real-time credit card fraud detection: supervised ML + deep learning models,
combined into a hybrid stacking ensemble, served through a Streamlit web app
deployed on Render.

**Dataset:** Kaggle's [Credit Card Transactions Fraud Detection Dataset](https://www.kaggle.com/datasets/kartik2112/fraud-detection)
(kartik2112/fraud-detection) — ~1.85M Sparkov-simulated transactions,
January 2019 – December 2020, ~0.58% fraud rate, shipped as `fraudTrain.csv`
and `fraudTest.csv`.

**Headline result:** a hybrid stacking ensemble — tuned XGBoost, Random
Forest, a feedforward neural network, and an LSTM, combined through a
Logistic Regression meta-learner — beats every individual model on both
validation (PR-AUC 0.9829) and the untouched final test holdout
(PR-AUC 0.9731). The meta-learner genuinely blends complementary signal
rather than just picking a winner, evidenced by all four base models
carrying non-trivial weight in its learned coefficients (see §7).

This notebook calls the project's own pipeline code in `src/` (imported as
modules, not reimplemented here) so the notebook and the production
scripts can never drift out of sync. Each expensive step is skipped if its
output already exists on disk — delete a specific `data/processed/`,
`models/`, or `reports/` file to force that one step to redo.

**Methodology discipline followed throughout:**
- Time-based (not random) train/validation/test splitting, to avoid a
  card's future transactions leaking into its own training data.
- Every "history-aware" feature (velocity, spending deviation, LSTM
  sequences) is computed strictly backward-looking.
- `fraudTest.csv` is touched exactly once, at the very end, for a single
  final evaluation of the already-chosen model.

In [ ]:
import sys
import warnings
from pathlib import Path

import joblib
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
plt.rcParams["figure.figsize"] = (9, 4.5)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

# Resolve the project's src/ directory regardless of whether this notebook
# is launched from notebooks/ (the usual case) or the project root.
_cwd = Path.cwd()
PROJECT_ROOT = _cwd.parent if _cwd.name == "notebooks" else _cwd
SRC_DIR = PROJECT_ROOT / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from config import MODELS_DIR, PROCESSED_DIR, RAW_TEST_CSV, RAW_TRAIN_CSV, REPORTS_DIR

print(f"Project root: {PROJECT_ROOT}")
print(f"Raw data present: {RAW_TRAIN_CSV.exists() and RAW_TEST_CSV.exists()}")
if not (RAW_TRAIN_CSV.exists() and RAW_TEST_CSV.exists()):
    print(
        "\\n!! Put Kaggle's fraudTrain.csv and fraudTest.csv in data/raw/ "
        "before running the cells below -- see README.md Setup."
    )

## 1. Data Split

`fraudTest.csv` is kept exactly as Kaggle provided it and used only as the
final holdout — never touched during model development. `fraudTrain.csv`
is sorted by timestamp and its most recent 15% is carved off as a
validation set (for tuning and model selection), with the earlier 85% as
training data. Carving validation from the *tail* chronologically, rather
than randomly, avoids letting a card's later transactions leak into
training while an earlier transaction from the same card sits in
validation.

In [ ]:
import data_split
from config import PROCESSED_TEST_CSV, PROCESSED_TRAIN_CSV, PROCESSED_VAL_CSV

if not PROCESSED_TRAIN_CSV.exists():
    data_split.main()
else:
    print("Split already exists -- skipping (delete data/processed/{train,val,test}.csv to re-run).\n")

for name, path in [("train", PROCESSED_TRAIN_CSV), ("val", PROCESSED_VAL_CSV), ("test", PROCESSED_TEST_CSV)]:
    split_df = pd.read_csv(path, usecols=["is_fraud"])
    print(f"{name:>5}: {len(split_df):>10,} rows, {split_df['is_fraud'].mean():.4%} fraud")

## 2. EDA and Feature Engineering

Exploratory analysis (`notebooks/01_eda.ipynb`) surfaced the findings that
shaped every feature below:

- **Fraud clusters heavily overnight**, roughly 10pm–3am — motivating an
  explicit `is_night` flag (plotted below, computed live from this run's
  training split).
- **Spending deviation was the strongest single signal found**: a
  transaction's dollar amount relative to a card's own historical mean
  spend, as a z-score. Fraudulent transactions showed a median z-score of
  roughly 3.1 standard deviations above a card's typical spend, versus
  roughly 0 for legitimate ones.
- **Transaction velocity** (count and dollar-sum of a card's transactions
  in the trailing 1h/24h) also showed strong separation.
- **Cardholder-to-merchant distance showed almost no separation** (~47
  miles on average either way) — kept anyway since it costs tree models
  nothing, but not expected to carry weight.
- A **mild fraud-rate difference by cardholder gender** was noted as a
  fairness consideration worth flagging rather than engineering around.

All features that look at a card's past behavior are computed strictly
backward-looking (`closed="left"` rolling windows, or an explicit
`.shift(1)` before any expanding calculation) — see `src/features.py`.

In [ ]:
import build_features
from data_loader import FEATURES_TRAIN_CSV, load_engineered_train

if not FEATURES_TRAIN_CSV.exists():
    build_features.main()
else:
    print("Engineered features already cached -- skipping (delete data/processed/*_features.csv to re-run).\n")

train_fe = load_engineered_train()
train_fe.head()

In [ ]:
# Reproduce the "fraud clusters overnight" EDA finding directly from this run's data.
fraud_rate_by_hour = train_fe.groupby("hour")["is_fraud"].mean()
ax = fraud_rate_by_hour.plot(kind="bar", color="#c0392b")
ax.set_title("Fraud rate by hour of day (training split)")
ax.set_xlabel("Hour of day")
ax.set_ylabel("Fraud rate")
plt.tight_layout()
plt.show()

**Engineered features:**

| Feature | Description |
| --- | --- |
| `distance_mi` | Haversine distance between cardholder and merchant location |
| `hour`, `day_of_week`, `is_weekend`, `month` | Calendar features from the transaction timestamp |
| `is_night` | Flag for the 10pm–3am window EDA identified as fraud-heavy |
| `hour_sin/cos`, `dow_sin/cos` | Cyclical encodings of hour and day-of-week |
| `customer_age` | Age at time of transaction, from date of birth |
| `city_pop_log` | Log-scaled cardholder city population |
| `txn_count_1h/24h`, `txn_amt_sum_1h/24h` | Rolling transaction count and dollar-sum for the same card in the trailing 1h/24h |
| `amt_zscore_vs_card_history` | This transaction's amount as a z-score against the card's own prior mean/std spend |

High-cardinality identifiers (card number, merchant name, job, street,
city, state, zip, names, transaction number) are deliberately excluded
from the modeling feature set — see `src/preprocessing.py`.

## 3. Baseline Models

Four baselines, each handling the ~0.58% fraud rate explicitly rather than
relying on raw accuracy: Logistic Regression and Random Forest use
`class_weight="balanced"`; XGBoost uses `scale_pos_weight`; a fourth
variant compares SMOTE oversampling against plain class-weighting for
Logistic Regression.

**PR-AUC (average precision) is the primary ranking metric throughout** —
at this level of imbalance, ROC-AUC is misleadingly high for every model
and doesn't reflect the precision/recall trade-off the way PR-AUC does.

In [ ]:
import train_baselines

baseline_results_path = REPORTS_DIR / "milestone3_model_comparison.csv"
if not baseline_results_path.exists():
    train_baselines.main()
else:
    print("Baseline results already exist -- skipping (delete the file above to re-run).\n")

baseline_df = pd.read_csv(baseline_results_path).sort_values("pr_auc", ascending=False).reset_index(drop=True)
baseline_df

In [ ]:
ax = baseline_df.set_index("model")["pr_auc"].plot(kind="barh", color="#2c3e50")
ax.set_xlabel("PR-AUC (validation)")
ax.set_title("Baseline models")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 4. Hyperparameter Tuning

`RandomizedSearchCV` (25 iterations, scoring on average precision) over
XGBoost and Random Forest, using a single-fold `PredefinedSplit` — a
stratified subsample of train for fitting, scored against the full
validation set — rather than full k-fold cross-validation, which would be
impractical at ~1.1M training rows. Best parameters are then refit on the
complete training set. This is a deliberate speed/rigor trade-off.

In [ ]:
import tune_models

tuning_results_path = REPORTS_DIR / "milestone3b_tuning_comparison.csv"
if not tuning_results_path.exists():
    tune_models.main()
else:
    print("Tuning results already exist -- skipping (delete the file above to re-run).\n")

tuning_df = pd.read_csv(tuning_results_path).sort_values("pr_auc", ascending=False).reset_index(drop=True)
tuning_df

In [ ]:
best_params_path = REPORTS_DIR / "milestone3b_best_params.csv"
if best_params_path.exists():
    display(pd.read_csv(best_params_path))

**Finding worth reporting honestly:** tuning did not clearly beat the
untuned baselines for either model — both land within noise of their
baseline counterparts, suggesting the defaults were already close to a
local optimum for this data (or that this search budget wasn't enough to
find a better one). The hybrid ensemble below accordingly uses the
**baseline** Random Forest, not the tuned one.

## 5. Deep Learning — Feedforward Network (FNN)

A PyTorch MLP: three hidden layers (128–64–32), each with BatchNorm, ReLU,
and dropout (0.3). Trained with `BCEWithLogitsLoss` (class-weighted via
`pos_weight`), Adam, `ReduceLROnPlateau`, and early stopping on validation
PR-AUC (patience 6, max 40 epochs).

In [ ]:
import train_fnn

fnn_result_path = REPORTS_DIR / "milestone4_fnn_result.csv"
if not fnn_result_path.exists():
    train_fnn.main()
else:
    print("FNN already trained -- skipping (delete the file above, and models/fnn_best.pt, to re-run).\n")

fnn_history = pd.read_csv(REPORTS_DIR / "milestone4_fnn_training_history.csv")
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(fnn_history["epoch"], fnn_history["train_loss"], color="#2c3e50")
axes[0].set_title("FNN training loss")
axes[0].set_xlabel("Epoch")
axes[1].plot(fnn_history["epoch"], fnn_history["val_pr_auc"], color="#c0392b")
axes[1].set_title("FNN validation PR-AUC")
axes[1].set_xlabel("Epoch")
plt.tight_layout()
plt.show()

pd.read_csv(fnn_result_path)

## 6. Deep Learning — LSTM

Unlike every model so far, the LSTM sees each transaction as the end of a
short sequence of that card's recent activity. A two-branch architecture:
the preceding 5 transactions' features run through an LSTM layer, whose
final hidden state is concatenated with static per-transaction context
(customer age, city population, category, gender) before a small
classification head. Sequences are built via vectorized
`groupby(card).shift(lag)` — the approach that scales to the full
1.85M-row dataset without a dedicated sequence library. Cards with fewer
than 5 prior transactions are zero-padded, a simplification worth naming
explicitly.

In [ ]:
import build_sequences

if not (PROCESSED_DIR / "train_sequences.npz").exists():
    build_sequences.main()
else:
    print("Sequences already built -- skipping (delete data/processed/*_sequences.npz to re-run).\n")

In [ ]:
import train_lstm

lstm_result_path = REPORTS_DIR / "milestone4_lstm_result.csv"
if not lstm_result_path.exists():
    train_lstm.main()
else:
    print("LSTM already trained -- skipping (delete the file above, and models/lstm_best.pt, to re-run).\n")

lstm_history = pd.read_csv(REPORTS_DIR / "milestone4_lstm_training_history.csv")
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].plot(lstm_history["epoch"], lstm_history["train_loss"], color="#2c3e50")
axes[0].set_title("LSTM training loss")
axes[0].set_xlabel("Epoch")
axes[1].plot(lstm_history["epoch"], lstm_history["val_pr_auc"], color="#c0392b")
axes[1].set_title("LSTM validation PR-AUC")
axes[1].set_xlabel("Epoch")
plt.tight_layout()
plt.show()

pd.read_csv(lstm_result_path)

## 7. The Hybrid Framework — Stacking Ensemble

This is the project's central creative contribution: a **stacking
ensemble** combining all four models above through a Logistic Regression
meta-learner trained on their predicted probabilities. Rather than
averaging the four models' outputs or deploying only the single best one,
the meta-learner *learns how much to trust each model*, extracting
complementary signal from models that disagree on different transactions.

**Evaluation design.** The base models were already selected/tuned using
validation, so fitting *and* scoring the meta-learner on validation too
would let it overfit trivially. Instead: **5-fold cross-validation within
validation** generates out-of-fold (OOF) meta-predictions — each
validation row's hybrid prediction comes from a meta-learner that never
saw that row during its own fit — giving an honest hybrid score on the
*full* validation set. A separate final meta-learner, fit on all of
validation, is what's actually saved and deployed.

In [ ]:
import train_hybrid

hybrid_result_path = REPORTS_DIR / "milestone4_hybrid_result.csv"
if not hybrid_result_path.exists():
    train_hybrid.main()
else:
    print("Hybrid already trained -- skipping (delete the file above, and models/hybrid_meta_learner.joblib, to re-run).\n")

pd.read_csv(hybrid_result_path)

In [ ]:
# The deployed meta-learner's own learned coefficients (fit on all of
# validation) -- how much it trusts each base model in the final blend.
hybrid_bundle = joblib.load(MODELS_DIR / "hybrid_meta_learner.joblib")
coefs = pd.Series(
    dict(zip(hybrid_bundle["base_model_order"], hybrid_bundle["meta_learner"].coef_[0]))
).sort_values(ascending=False)

ax = coefs.plot(kind="barh", color="#8e44ad")
ax.set_title("Hybrid meta-learner: learned trust per base model")
ax.set_xlabel("Logistic Regression coefficient")
ax.invert_yaxis()
plt.tight_layout()
plt.show()
coefs

All four base models carry non-trivial, non-zero weight in the blend —
evidence the ensemble is genuinely combining complementary signal, not
just amplifying whichever single model happens to be strongest.

## 8. Full Validation Leaderboard

Every model trained above, aggregated and ranked by validation PR-AUC.

In [ ]:
leaderboard_df = pd.read_csv(REPORTS_DIR / "leaderboard.csv").sort_values("pr_auc", ascending=False).reset_index(drop=True)
display(leaderboard_df)

ax = leaderboard_df.set_index("model")["pr_auc"].plot(kind="barh", color="#16a085")
ax.set_xlabel("PR-AUC (validation)")
ax.set_title("Full leaderboard")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## 9. Final Test-Holdout Evaluation

`fraudTest.csv` has been untouched since §1. It is scored **exactly once**,
after the final model (the hybrid) was chosen using validation alone —
scoring it repeatedly and reporting whichever run looks best would quietly
turn it into a second validation set. `src/evaluate_final_test.py` enforces
this: it refuses to re-run if a result already exists, and it reports
performance at each model's **validation-chosen threshold** (the one that
would actually ship), never a threshold re-tuned on test itself.

This notebook does not call that script automatically, to avoid ever
triggering the holdout evaluation as a side effect of re-running a
notebook cell. Run it yourself, once, from the command line:

```bash
python src/evaluate_final_test.py
```

The cell below simply loads the result if it exists.

In [ ]:
final_test_path = REPORTS_DIR / "final_test_evaluation.csv"

if final_test_path.exists():
    final_test_df = pd.read_csv(final_test_path).sort_values("pr_auc", ascending=False).reset_index(drop=True)
    val_pr_auc = leaderboard_df.set_index("model")["pr_auc"]
    final_test_df["val_pr_auc"] = final_test_df["model"].map(val_pr_auc)
    final_test_df["gap"] = final_test_df["pr_auc"] - final_test_df["val_pr_auc"]
    display(
        final_test_df[
            ["model", "pr_auc", "val_pr_auc", "gap", "shipped_precision", "shipped_recall", "shipped_f1"]
        ].rename(columns={"pr_auc": "test_pr_auc"})
    )
else:
    print(
        "No final_test_evaluation.csv yet. Run `python src/evaluate_final_test.py` "
        "from the command line once you're ready to report a final number, then "
        "re-run this cell."
    )

## 10. Web Application and Deployment

The chosen hybrid pipeline is served through a Streamlit web app
(`app/app.py`) that scores uploaded transaction batches or a single
manually-entered transaction, and is deployed live on Render
(`render.yaml`, `requirements-render.txt`). See the project README for
setup, the app's documented limitation around per-card transaction
history, and deployment details — out of scope for this modeling
notebook.

## 11. Limitations and Future Work

- **No live transaction history in the web app** — velocity,
  spending-deviation, and sequence features fall back to neutral defaults
  without an uploaded batch providing real history; a production system
  would need a real-time per-card history store.
- **Zero-padded LSTM sequences**, rather than packed sequences with
  masking — simpler and sufficient here, but worth revisiting for
  production.
- **Simulated data.** The Sparkov-generated dataset does not capture the
  full complexity and adversarial adaptation of real-world fraud; a
  production model would need continuous retraining against evolving
  fraud tactics.
- **Random Forest's uncapped tree depth** produces a large model
  artifact; a depth cap would likely shrink it with negligible accuracy
  cost.
- **Fairness.** A mild fraud-rate difference by cardholder gender was
  noted during EDA but not investigated further; a production deployment
  would warrant a fuller fairness audit before relying on model outputs
  to affect real customers.

## Conclusion

The hybrid stacking ensemble — tuned XGBoost, Random Forest, an FNN, and
an LSTM combined through a Logistic Regression meta-learner — is the
project's best model on both validation (PR-AUC 0.9829) and the untouched
final test holdout (PR-AUC 0.9731), and it generalized more reliably than
its strongest individual component: the LSTM scored second-best on
validation but fell to fourth on test, while the hybrid's validation
score held up as an honest predictor of real-world performance. That
robustness — not just the raw PR-AUC gain — is the strongest argument for
shipping the hybrid over any single model, and it is what actually runs
in the deployed web application.